# STARRA Voice Design Kit — Qwen3-TTS VoiceDesign

Runs top-to-bottom on a Colab **GPU** runtime. Designs a voice from a text
description only, lets you iterate the description until she sounds right,
then exports 48kHz WAV assets for drop-in use in the STARGATE deck.

**HARD ETHICS RULE:** this notebook only does *voice design* — a voice
synthesized from a natural-language description. It never loads a reference
audio clip of a real person and never calls Qwen3-TTS's voice-cloning path
(`generate_voice_clone` / `ref_audio` + `ref_text`). Do not add one.

**Sources cited in this notebook** (fetched 2026-08-04):
- Repo: https://github.com/QwenLM/Qwen3-TTS — README (install commands,
  model ID table, `generate_voice_design()` example code), Apache-2.0
  license, last pushed 2026-03-17T06:38:41Z (verified via
  `api.github.com/repos/QwenLM/Qwen3-TTS`).
- Model card: https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign —
  params, supported languages, license.

See `README.md` next to this notebook for how to run it, where the exported
files land in the STARGATE repo, and what consumes them now vs. later.


## 0. GPU runtime check

Qwen3-TTS-12Hz-1.7B needs a CUDA GPU to run at a usable speed for iteration
(CPU inference works but is impractically slow for a "type a line, listen"
loop). This cell fails fast with the fix if the runtime is CPU-only.


In [ ]:
import subprocess

def _has_gpu() -> bool:
    try:
        out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
        return out.returncode == 0
    except FileNotFoundError:
        return False

if not _has_gpu():
    raise RuntimeError(
        "No GPU detected in this Colab runtime.\n"
        "Fix: Runtime -> Change runtime type -> Hardware accelerator -> GPU "
        "(T4 is fine on the free tier; Colab Pro can assign a better GPU), "
        "then Runtime -> Restart session, then re-run this notebook from the top."
    )

print("GPU detected:")
print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
        capture_output=True, text=True,
    ).stdout
)


## 1. Install

Straight from the repo README (github.com/QwenLM/Qwen3-TTS, install section).
`qwen-tts` pulls its own `torch`/`transformers` as dependencies — the README
does not pin specific versions beyond that, so this notebook doesn't
second-guess it.

FlashAttention 2 is optional: it reduces VRAM and speeds up generation, but it
compiles from source and can be slow or flaky on some Colab GPU images. The
model-load cell below tries it and falls back cleanly to default attention if
it isn't installed, so leaving it uninstalled is a safe default.


In [ ]:
%pip install -q -U qwen-tts

# Optional FlashAttention 2 (github.com/QwenLM/Qwen3-TTS README):
#   pip install -U flash-attn --no-build-isolation
# If your Colab runtime has < 96GB RAM and many CPU cores, the README
# recommends capping parallel build jobs:
#   MAX_JOBS=4 pip install -U flash-attn --no-build-isolation
# Uncomment ONE of the next two lines to try it. Not required — the
# model-load cell below falls back to default attention if this is skipped.
# %pip install -q -U flash-attn --no-build-isolation
# %env MAX_JOBS=4
# %pip install -q -U flash-attn --no-build-isolation


## 2. Load the model

Model: `Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign` — the free-form voice-design
checkpoint (natural-language `instruct=` description, no reference audio).
Deliberately NOT `-CustomVoice` or `-Base`, which are the reference-audio /
cloning paths this kit avoids per the ethics rule above.

**VRAM — unverified estimate.** Neither the repo README nor the HF model
card states a VRAM figure for this checkpoint (model card gives only "2B
params", fetched 2026-08-04). Estimate: 1.7B params in bf16 is roughly 3.4GB
of weights alone; with the audio codec/tokenizer, KV cache, and generation
overhead, plan for **roughly 4-6GB actually resident** — comfortably inside a
Colab free-tier T4 (16GB), with room to spare. An A100 (Colab Pro) is not
needed for single-utterance voice design; it just runs faster. The cell below
prints the *actual measured* allocation right after load so you're not stuck
relying on this estimate.


In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"  # github.com/QwenLM/Qwen3-TTS README, model table

assert torch.cuda.is_available(), "CUDA not available — see the GPU runtime check cell above."

try:
    model = Qwen3TTSModel.from_pretrained(
        MODEL_ID,
        device_map="cuda:0",
        dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
    )
    print("Loaded with FlashAttention 2.")
except Exception as e:
    print(f"FlashAttention 2 unavailable ({e!r}); loading with default attention instead.")
    model = Qwen3TTSModel.from_pretrained(
        MODEL_ID,
        device_map="cuda:0",
        dtype=torch.bfloat16,
    )
    print("Loaded with default attention.")

print(f"GPU memory allocated after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 3. Voice description

Written as voice **quality** — timbre, pitch register, pace, warmth, accent —
not ethnicity checkboxes. `generate_voice_design(instruct=...)` takes
free-form natural language (repo README, "Voice Design" example); there is no
fixed vocabulary to match, so the wording below is deliberately descriptive
rather than categorical, aimed at the acoustic target Classic described:
early-20s woman, blended/mixed vocal timbre, British accent, warm Millennial
delivery.


In [ ]:
# Primary voice-design instruction. Edit this and re-run the ITERATE cell
# below to hear changes — nothing is saved until the EXPORT cell runs.
VOICE_DESCRIPTION = (
    "A woman in her early twenties, speaking with a contemporary British "
    "accent (South-East England, not cut-glass RP). Timbre: warm, rounded "
    "low-mid register with a soft breathy edge and smooth, clear consonants "
    "— not nasal, not thin, not bright or tinny. Medium-low pitch for a "
    "young woman's voice. Pace: unhurried and conversational, with natural "
    "pauses between thoughts — never rushed or clipped. Delivery: warm "
    "Millennial energy — casual confidence, a little playful, present and "
    "intimate, like talking to someone she's close to. Not corporate, not "
    "newsreader-formal, not robotic or over-enunciated."
)

# Alternative phrasings — swap VOICE_DESCRIPTION for one of these (or your own
# edit of one) and re-run ITERATE to compare. Pass instruct=ALT_DESCRIPTIONS["..."]
# to synth() without touching VOICE_DESCRIPTION itself.
ALT_DESCRIPTIONS = {
    "texture_forward": (
        "Early-20s British woman, unhurried conversational pace. Voice has a "
        "warm, slightly husky lower register blended with clear, precise "
        "consonant articulation — smooth resonance, not breathy-weak, not "
        "sharp or thin. Playful undertone, casual Millennial warmth, present "
        "tense energy, never stiff or sing-song."
    ),
    "warmth_forward": (
        "A warm, soft-spoken young British woman's voice, medium-low pitch, "
        "relaxed pace with real pauses and breath between thoughts. Rich, "
        "rounded tone with a light, clear edge on consonants — sounds close "
        "and intimate, like a good friend on the phone. Casual, present, "
        "unforced warmth; no corporate polish, no over-enunciation."
    ),
    "pace_forward": (
        "Twenty-something woman, contemporary British accent, confident and "
        "relaxed delivery, medium-slow pace with natural pauses. Warm "
        "chest-register resonance blended with a bright, clear articulate "
        "edge — sounds attentive and present, playful but grounded, never "
        "performative or robotic."
    ),
}

print(VOICE_DESCRIPTION)


## 4. Iterate

Type a line into `TEST_LINE`, run the cell, listen inline. Change
`VOICE_DESCRIPTION` above (or call `synth(text, instruct=ALT_DESCRIPTIONS["..."])`)
and re-run to compare takes. `synth()` resamples the model's native output to
48kHz so every clip in this notebook — test takes and exports alike — is the
same sample rate.


In [ ]:
import numpy as np
import scipy.signal
from IPython.display import Audio, display

TARGET_SR = 48000

def _to_48k(wav: np.ndarray, sr: int) -> np.ndarray:
    if sr == TARGET_SR:
        return wav.astype(np.float32)
    n_out = int(round(len(wav) * TARGET_SR / sr))
    return scipy.signal.resample(wav, n_out).astype(np.float32)

def synth(text: str, instruct: str = None, language: str = "English"):
    """Synthesize one line with the loaded VoiceDesign model. Returns (wav, sr) at 48kHz."""
    if instruct is None:
        instruct = VOICE_DESCRIPTION
    wavs, sr = model.generate_voice_design(text=text, language=language, instruct=instruct)
    wav = _to_48k(np.asarray(wavs[0], dtype=np.float32), sr)
    return wav, TARGET_SR

# --- Edit this line, run the cell, listen. Repeat. ---
TEST_LINE = "Evening, Classic."

wav, sr = synth(TEST_LINE)
display(Audio(wav, rate=sr))


## 5. Signature lines

Four lines, pulled from the live repo rather than invented — see the code
comment for exact citations. These get exported as individual clips in
Section 6.


In [ ]:
SIGNATURE_LINES_PATH = "/content/signature_lines.txt"

# Sources (verified in the STARGATE repo, 2026-08-04):
#  1. Boot greeting  — DEFAULT_BOOT_GREETING, src/lib/voice/deck-prefs.ts:26
#     ("What she says once the deck finishes booting, until Classic rewrites it.")
#  2. Operator ack    — "Copy.", repo CLAUDE.md ("Operator voice: 'Copy.' not 'Sure!'")
#  3. Call-opening    — representative example matching her persona.ts canon
#     (IDENTITY/PRESENCE blocks, src/lib/persona.ts). NOTE: there is no single
#     fixed call-opening string to extract — real openings are generated live
#     by the brain per call (src/lib/voice/useVoiceCall.ts:1710-1712, the
#     "[The voice call just connected — greet him ...]" stage direction), so
#     this line is a representative stand-in for voice-testing purposes, not
#     a literal quote from the code.
#  4. Ringback line   — a short spoken tease to pair with / replace the
#     current ringback, which today is an instrumental clip, not speech
#     (public/avatar/starra-ringback.mp3, played by VoiceCallOverlay.tsx).
SIGNATURE_LINES = [
    "Evening, Classic.",
    "Copy.",
    "Hey, Classic -- good to hear you. What are we building today?",
    "Hang on, Classic -- I'm picking up right now.",
]

with open(SIGNATURE_LINES_PATH, "w") as f:
    f.write("\n".join(SIGNATURE_LINES) + "\n")

print(f"Wrote {len(SIGNATURE_LINES)} lines to {SIGNATURE_LINES_PATH}:")
for line in SIGNATURE_LINES:
    print(" -", line)


## 6. Export

Produces one ~30s neutral reference read (`00_reference.wav`) plus one clip
per signature line (in the order written above), all at 48kHz PCM16, zipped,
and downloaded via `files.download()`. See `README.md` for exactly where each
file lands in the STARGATE repo and what to do with it.


In [ ]:
import os
import zipfile
from datetime import datetime

import soundfile as sf

OUT_DIR = "/content/voice_export"
os.makedirs(OUT_DIR, exist_ok=True)

# ~30s neutral reference read. Exact duration depends on the model's own
# pacing for this instruct description -- the printed duration below tells
# you how close it landed; trim/pad this text and re-run if it's off.
NEUTRAL_REFERENCE_TEXT = (
    "This is a short reference recording, used to check tone, pace, and "
    "clarity before anything else gets built on top of it. The quick brown "
    "fox jumps over the lazy dog, and the five boxing wizards jump quickly. "
    "Numbers like twelve, forty, and one hundred and three should sound "
    "natural, and so should everyday words: kitchen, Tuesday, umbrella, "
    "October. At a relaxed, unhurried pace, with clear consonants and "
    "natural pauses between thoughts, this reading should run close to "
    "thirty seconds from start to finish."
)

def _write_clip(name: str, text: str) -> str:
    wav, sr = synth(text)
    path = os.path.join(OUT_DIR, name)
    sf.write(path, wav, sr, subtype="PCM_16")
    print(f"{name}: {len(wav) / sr:.1f}s")
    return path

exported = [_write_clip("00_reference.wav", NEUTRAL_REFERENCE_TEXT)]

_names = [
    "01_boot_greeting.wav",
    "02_operator_ack.wav",
    "03_call_opening.wav",
    "04_ringback_line.wav",
]
for name, line in zip(_names, SIGNATURE_LINES):
    exported.append(_write_clip(name, line))

zip_path = f"/content/starra-voice-{datetime.now():%Y%m%d-%H%M%S}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in exported:
        zf.write(path, arcname=os.path.basename(path))

print(f"\nZipped {len(exported)} files -> {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab -- files are on disk at:", zip_path)


## Next

Download the zip, unpack it, and see `README.md` in this folder for where
each file goes in the STARGATE repo (drop-in replacement paths) and how to
convert `04_ringback_line.wav` to mp3 locally for the existing ringback
asset slot.
